# Data Download: Replicating Gu, Kelly & Xiu (2020)
## "Empirical Asset Pricing via Machine Learning" — Review of Financial Studies

This notebook downloads and prepares the data needed for the cross-sectional replication.

**Data components:**
1. **94 firm-level characteristics** (from Dacheng Xiu's website or JKP/WRDS)
2. **Monthly stock returns** (from CRSP via WRDS)
3. **8 macroeconomic predictors** (Welch & Goyal 2008)
4. **74 SIC-2 industry dummies** (constructed from CRSP SIC codes)

**Total covariates:** 94 × (8 + 1) + 74 = **920** baseline signals

---

## 0. Install Dependencies

In [1]:
# Run this cell once to install required packages
!pip install requests tqdm

In [5]:
import pandas as pd
import numpy as np
import os
import io
import zipfile
import requests
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Create a data directory
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

print('Setup complete.')

Setup complete.


---
## 1. Firm-Level Characteristics (94 predictors)

### Direct download from Dacheng Xiu's homepage 
The authors provide the pre-computed dataset of 94 lagged firm characteristics at:

**`https://dachxiu.chicagobooth.edu/download/datashare.zip`**

This file is ~3-4 GB and contains `datashare.csv` with:
- `DATE`: end of month (YYYYMMDD), already lag-adjusted
- `permno`: CRSP permanent company number
- 94 columns of firm characteristics

> **Note from the readme:** *"In this dataset, we've already adjusted the lags. (e.g. When DATE=19570329, you can use the monthly RET at 195703 as the response variable.)"*

In [6]:
# ====================================================
# Download characteristics from Xiu's website
# ====================================================
# WARNING: This file is ~3-4 GB. 

url_chars = 'https://dachxiu.chicagobooth.edu/download/datashare.zip'
zip_path = DATA_DIR / 'datashare.zip'
csv_path = DATA_DIR / 'datashare.csv'

if not csv_path.exists():
    print('Downloading firm characteristics from Dacheng Xiu homepage...')
    print('This may take 10-20 minutes depending on your connection.')
    
    # Stream download to handle large file
    with requests.get(url_chars, stream=True, timeout=1200) as r:
        r.raise_for_status()
        total_size = int(r.headers.get('content-length', 0))
        downloaded = 0
        with open(zip_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192*100):
                f.write(chunk)
                downloaded += len(chunk)
                if total_size > 0:
                    pct = downloaded / total_size * 100
                    print(f'\rProgress: {pct:.1f}% ({downloaded/1e9:.2f} GB)', end='')
    print('\nDownload complete. Extracting...')
    
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    print('Extraction complete.')
    
    # Clean up zip
    zip_path.unlink()
else:
    print(f'File already exists: {csv_path}')

File already exists: data/datashare.csv


In [7]:
# Load the characteristics data
csv_path = DATA_DIR / 'datashare.csv'
print('Loading characteristics data (this may take a minute)...')
chars = pd.read_csv(csv_path, low_memory=False)

# Basic formatting
chars.rename(columns={'DATE': 'date', 'permno': 'permno'}, inplace=True)
chars['date'] = pd.to_datetime(chars['date'], format='%Y%m%d')
chars['date'] = chars['date'].dt.to_period('M')  # Convert to monthly period

print(f'Shape: {chars.shape}')
print(f'Date range: {chars["date"].min()} to {chars["date"].max()}')
print(f'Unique stocks (permno): {chars["permno"].nunique()}')
print(f'\nCharacteristics ({chars.shape[1] - 2} features):')
print([c for c in chars.columns if c not in ['date', 'permno']])

Loading characteristics data (this may take a minute)...
Shape: (4117300, 97)
Date range: 1957-01 to 2021-12
Unique stocks (permno): 32793

Characteristics (95 features):
['mvel1', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom', 'mom1m', 'mom6m', 'mom12m', 'mom36m', 'pricedelay', 'turn', 'absacc', 'acc', 'age', 'agr', 'bm', 'bm_ia', 'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho', 'chempia', 'chinv', 'chpmia', 'convind', 'currat', 'depr', 'divi', 'divo', 'dy', 'egr', 'ep', 'gma', 'grcapx', 'grltnoa', 'herf', 'hire', 'invest', 'lev', 'lgr', 'mve_ia', 'operprof', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr', 'pchgm_pchsale', 'pchquick', 'pchsale_pchinvt', 'pchsale_pchrect', 'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'ps', 'quick', 'rd', 'rd_mve', 'rd_sale', 'realestate', 'roic', 'salecash', 'saleinv', 'salerec', 'secured', 'securedind', 'sgr', 'sin', 'sp', 'tang', 'tb', 'aeavol', 'cash', 'chtx', 'cinvest', 'ear', 'nincr', 'roaq', 'roavol', 'roeq', 'rsup', 'stdacc',

In [8]:
# Quick summary stats
chars.describe().T.head(20)

,count,mean,std,min,25%,50%,75%,max
permno,4117300.0,5.580153e+04,2.803699e+04,10000.000000,27748.000000,61321.000000,81002.000000,9.343600e+04
mvel1,4114230.0,1.794745e+06,1.425696e+07,0.000000,22628.333804,99787.901182,500489.379515,2.711977e+09
beta,3716736.0,1.017846e+00,6.469574e-01,-0.799580,0.551429,0.948765,1.393751,3.887420e+00
betasq,3716736.0,1.459436e+00,1.727259e+00,0.000000,0.309960,0.903993,1.946645,1.511203e+01
chmom,3778195.0,1.419177e-03,5.388882e-01,-8.461765,-0.237329,-0.005600,0.231787,7.783077e+00
dolvol,3765378.0,1.103710e+01,2.997975e+00,-3.060271,8.932708,10.910054,13.119557,1.945995e+01
idiovol,3716736.0,6.000247e-02,3.719688e-02,0.000000,0.033552,0.050516,0.076566,2.835521e-01
indmom,4117276.0,1.290215e-01,2.796336e-01,-0.759503,-0.039005,0.105214,0.255788,2.748655e+00
mom1m,4085562.0,9.047752e-03,1.470126e-01,-0.736589,-0.060606,0.000000,0.065848,2.000000e+00
mom6m,3960127.0,4.944756e-02,3.507820e-01,-1.000000,-0.133818,0.022727,0.180602,7.533333e+00


---
## 2. Monthly Stock Returns from CRSP

We need CRSP monthly returns to compute excess returns (the target variable). 
This requires WRDS access.

**From the paper (p. 2248):**
> We obtain monthly total individual equity returns from CRSP for all firms listed in the NYSE, AMEX, and NASDAQ. Our sample begins in March 1957.

In [9]:
import wrds

# Connect to WRDS
wrds_db = wrds.Connection()
print('Connected to WRDS.')

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Connected to WRDS.


In [10]:
# ====================================================
# Download CRSP Monthly Stock File
# ====================================================
# We get: permno, date, ret (total return), shrout, prc (price), 
# exchcd (exchange code), siccd (SIC code), shrcd (share code)

crsp_query = """
    SELECT a.permno, a.date, a.ret, a.retx, a.prc, a.shrout, a.vol,
           b.exchcd, b.siccd, b.shrcd
    FROM crsp.msf AS a
    LEFT JOIN crsp.msenames AS b
        ON a.permno = b.permno
        AND a.date >= b.namedt
        AND a.date <= b.nameendt
    WHERE a.date >= '1957-03-01'
      AND a.date <= '2021-12-31'
      AND b.exchcd IN (1, 2, 3)       -- NYSE, AMEX, NASDAQ
      AND b.shrcd IN (10, 11)          -- Common shares only
"""

print('Downloading CRSP monthly stock data...')
crsp_monthly = wrds_db.raw_sql(crsp_query, date_cols=['date'])
print(f'CRSP data shape: {crsp_monthly.shape}')
print(f'Date range: {crsp_monthly["date"].min()} to {crsp_monthly["date"].max()}')
print(f'Unique permnos: {crsp_monthly["permno"].nunique()}')

CRSP data shape: (3352317, 10)
Date range: 1957-03-29 00:00:00 to 2021-12-31 00:00:00
Unique permnos: 25845


In [12]:
# ====================================================
# Download CRSP delisting returns
# ====================================================
# Important: incorporate delisting returns to avoid survivorship bias

delist_query = """
    SELECT permno, dlstdt AS date, dlret, dlstcd
    FROM crsp.msedelist
    WHERE dlstdt >= '1957-03-01'
      AND dlstdt <= '2021-12-31'
"""

print('Downloading delisting returns...')
crsp_delist = wrds_db.raw_sql(delist_query, date_cols=['date'])
print(f'Delisting data shape: {crsp_delist.shape}')

Delisting data shape: (26414, 4)


In [13]:
# ====================================================
# Merge delisting returns with monthly returns
# ====================================================

# Align delisting date to end of month
crsp_delist['date'] = crsp_delist['date'] + pd.offsets.MonthEnd(0)
crsp_monthly['date'] = crsp_monthly['date'] + pd.offsets.MonthEnd(0)

# Merge
crsp = crsp_monthly.merge(crsp_delist[['permno', 'date', 'dlret', 'dlstcd']], 
                           on=['permno', 'date'], how='left')

# Adjust returns for delisting
# If dlret is missing and dlstcd indicates performance-related delisting, use -0.30
crsp.loc[
    (crsp['dlret'].isna()) & (crsp['dlstcd'].isin([500, 520, 551, 552, 560, 574])),
    'dlret'
] = -0.30

# Fill remaining missing dlret with 0 (non-performance delistings or no delisting)
crsp['dlret'] = crsp['dlret'].fillna(0)

# Compute adjusted return: (1 + ret) * (1 + dlret) - 1
crsp['ret_adj'] = (1 + crsp['ret'].fillna(0)) * (1 + crsp['dlret'].fillna(0)) - 1

# Market equity (in millions)
crsp['me'] = np.abs(crsp['prc']) * crsp['shrout'] / 1000

print(f'Merged CRSP shape: {crsp.shape}')
crsp.head()

Merged CRSP shape: (3352317, 14)


,permno,date,ret,retx,prc,shrout,vol,exchcd,siccd,shrcd,dlret,dlstcd,ret_adj,me
0,10000,1986-01-31,<NA>,<NA>,-4.375,3680.0,1771.0,3,3990,10,0.0,<NA>,0.0,16.1
1,10000,1986-02-28,-0.257143,-0.257143,-3.25,3680.0,828.0,3,3990,10,0.0,<NA>,-0.257143,11.96
2,10000,1986-03-31,0.365385,0.365385,-4.4375,3680.0,1078.0,3,3990,10,0.0,<NA>,0.365385,16.33
3,10000,1986-04-30,-0.098592,-0.098592,-4.0,3793.0,957.0,3,3990,10,0.0,<NA>,-0.098592,15.172
4,10000,1986-05-31,-0.222656,-0.222656,-3.10938,3793.0,1074.0,3,3990,10,0.0,<NA>,-0.222656,11.793878


In [14]:
# ====================================================
# Download risk-free rate from CRSP
# ====================================================

rf_query = """
    SELECT caldt AS date, t30ret AS rf
    FROM crsp.mcti
    WHERE caldt >= '1957-03-01'
      AND caldt <= '2021-12-31'
"""

print('Downloading risk-free rate...')
rf = wrds_db.raw_sql(rf_query, date_cols=['date'])
rf['date'] = rf['date'] + pd.offsets.MonthEnd(0)
print(f'Risk-free rate data shape: {rf.shape}')

Risk-free rate data shape: (778, 2)


In [15]:
# ====================================================
# Compute excess returns
# ====================================================

crsp = crsp.merge(rf, on='date', how='left')
crsp['ret_excess'] = crsp['ret_adj'] - crsp['rf']

print(f'Excess return stats:')
print(crsp['ret_excess'].describe())

Excess return stats:
count    3352317.0
mean      0.007911
std       0.180718
min      -1.014149
25%      -0.068695
50%      -0.004031
75%        0.06649
max      23.996941
Name: ret_excess, dtype: Float64


In [16]:
# Convert date to monthly period for merging with characteristics
crsp['date'] = crsp['date'].dt.to_period('M')

# Keep only the columns needed downstream
crsp = crsp[['permno', 'date', 'ret_excess', 'me', 'siccd', 'exchcd']].copy()
crsp = crsp.dropna(subset=['ret_excess'])

print(f'Clean CRSP shape: {crsp.shape}')
crsp.head()

Clean CRSP shape: (3352317, 6)


,permno,date,ret_excess,me,siccd,exchcd
0,10000,1986-01,-0.004865,16.1,3990,3
1,10000,1986-02,-0.262439,11.96,3990,3
2,10000,1986-03,0.359422,16.33,3990,3
3,10000,1986-04,-0.103933,15.172,3990,3
4,10000,1986-05,-0.22759,11.793878,3990,3


In [17]:
# Save to CSV
crsp.to_csv(DATA_DIR / "Monthly_returns_crsp.csv", index=False)
rf.to_csv(DATA_DIR / "risk_free_rate.csv", index=False)

## Variable definitions in Monthly_returns_crsp

- **permno**: Permanent CRSP identifier for each stock.
- **date**: Monthly observation date used for merging across datasets.
- **ret_excess**: Excess stock return over the risk-free rate, computed as \(\text{ret\_adj} - \text{rf}\).
- **me**: Market equity (market capitalization), computed as \(|prc| \times shrout / 1000\).
- **siccd**: Standard Industrial Classification (SIC) code identifying the firm's industry.
- **exchcd**: Exchange code indicating where the stock is listed (e.g. NYSE, AMEX, NASDAQ).

---
## 3. Macroeconomic Predictors (Welch & Goyal 2008)

The paper uses 8 aggregate time-series variables:
1. **dp** — Dividend-price ratio (log)
2. **ep** — Earnings-price ratio (log)
3. **bm** — Book-to-market ratio
4. **ntis** — Net equity expansion
5. **tbl** — Treasury bill rate
6. **tms** — Term spread
7. **dfy** — Default spread
8. **svar** — Stock variance

These are available from Amit Goyal's website.

In [18]:
# ====================================================
# Load Welch & Goyal (2008) macro predictors
# ====================================================
# Manual download required from: https://sites.google.com/view/agoyal145/home
# Save as: data/PredictorData.xlsx

macro_raw = pd.read_excel(DATA_DIR / 'PredictorData.xlsx', sheet_name='Monthly')
print(macro_raw.head())

   yyyymm  Index   D12  E12  b/m  tbl  AAA  BAA  lty  ntis     Rfree  infl  \
0  187101   4.44  0.26  0.4  NaN  NaN  NaN  NaN  NaN   NaN       NaN   NaN   
1  187102   4.50  0.26  0.4  NaN  NaN  NaN  NaN  NaN   NaN  0.004967   NaN   
2  187103   4.61  0.26  0.4  NaN  NaN  NaN  NaN  NaN   NaN  0.004525   NaN   
3  187104   4.74  0.26  0.4  NaN  NaN  NaN  NaN  NaN   NaN  0.004252   NaN   
4  187105   4.86  0.26  0.4  NaN  NaN  NaN  NaN  NaN   NaN  0.004643   NaN   

   ltr  corpr  svar  csp  CRSP_SPvw  CRSP_SPvwx  
0  NaN    NaN   NaN  NaN        NaN         NaN  
1  NaN    NaN   NaN  NaN        NaN         NaN  
2  NaN    NaN   NaN  NaN        NaN         NaN  
3  NaN    NaN   NaN  NaN        NaN         NaN  
4  NaN    NaN   NaN  NaN        NaN         NaN  


In [19]:
# ====================================================
# Process macro predictors
# ====================================================

macro = macro_raw.copy()

# Identify date column
date_col = [c for c in macro.columns if 'date' in c.lower() or 'yyyymm' in c.lower()]
if not date_col:
    date_col = [macro.columns[0]]
date_col = date_col[0]

macro[date_col] = macro[date_col].astype(str)
macro = macro[macro[date_col].str.len() >= 6].copy()
macro['date'] = pd.to_datetime(macro[date_col].str[:6], format='%Y%m').dt.to_period('M')

print('Available columns in macro data:')
print(macro.columns.tolist())

# Construct the 8 predictors used in GKX (2020)
macro_predictors = pd.DataFrame()
macro_predictors['date'] = macro['date']

# dp: log dividend-price ratio
macro_predictors['dp'] = np.log(macro['D12'].astype(float)) - np.log(macro['Index'].astype(float))
# ep: log earnings-price ratio
macro_predictors['ep'] = np.log(macro['E12'].astype(float)) - np.log(macro['Index'].astype(float))
# bm: book-to-market
macro_predictors['bm'] = macro['b/m'].astype(float)
# ntis: net equity expansion
macro_predictors['ntis'] = macro['ntis'].astype(float)
# tbl: Treasury bill rate
macro_predictors['tbl'] = macro['tbl'].astype(float)
# tms: term spread (long-term yield - T-bill)
macro_predictors['tms'] = macro['lty'].astype(float) - macro['tbl'].astype(float)
# dfy: default spread (BAA - AAA)
macro_predictors['dfy'] = macro['BAA'].astype(float) - macro['AAA'].astype(float)
# svar: stock variance
macro_predictors['svar'] = macro['svar'].astype(float)

print(f'\nMacro predictors shape: {macro_predictors.shape}')
print(macro_predictors.describe())

macro_predictors.to_csv(DATA_DIR / "Macro_Predictors.csv", index=False)

Available columns in macro data:
['yyyymm', 'Index', 'D12', 'E12', 'b/m', 'tbl', 'AAA', 'BAA', 'lty', 'ntis', 'Rfree', 'infl', 'ltr', 'corpr', 'svar', 'csp', 'CRSP_SPvw', 'CRSP_SPvwx', 'date']

Macro predictors shape: (1848, 9)
                dp           ep           bm         ntis          tbl  \
count  1848.000000  1848.000000  1246.000000  1177.000000  1260.000000   
mean     -3.259051    -2.697558     0.543384     0.015331     0.033608   
std       0.466256     0.385916     0.263315     0.025841     0.029543   
min      -4.523640    -4.836478     0.120510    -0.055954     0.000100   
25%      -3.501137    -2.918823     0.323685     0.002195     0.007475   
50%      -3.167254    -2.708179     0.523083     0.015588     0.030400   
75%      -2.930407    -2.447177     0.724794     0.026702     0.050825   
max      -1.873246    -1.670063     2.028478     0.177041     0.163000   

               tms          dfy         svar  
count  1260.000000  1272.000000  1679.000000  
mean      0

---
## 4. Merge Everything & Construct Final Dataset

In [20]:
# ====================================================
# Prepare variables and merge characteristics with returns
# ====================================================

# Define characteristic columns (exclude IDs and sic2)
char_exclude = ["permno", "date", "sic2"]
char_cols = [c for c in chars.columns if c not in char_exclude]

# Rename macro columns to avoid collisions with characteristic names (ep, bm)
macro_predictors = macro_predictors.rename(columns={
    "dp": "dp_macro", "ep": "ep_macro", "bm": "bm_macro",
    "ntis": "ntis_macro", "tbl": "tbl_macro", "tms": "tms_macro",
    "dfy": "dfy_macro", "svar": "svar_macro",
})

macro_vars = [
    "dp_macro", "ep_macro", "bm_macro", "ntis_macro",
    "tbl_macro", "tms_macro", "dfy_macro", "svar_macro"
]

# The characteristics are already lagged — the date in datashare.csv
# corresponds to the month whose return is the response variable.
# So we merge on permno and date directly.
base_data = chars.merge(
    crsp,
    on=["permno", "date"],
    how="inner",
    validate="one_to_one"
)

print(f"chars shape: {chars.shape}")
print(f"crsp shape: {crsp.shape}")
print(f"Number of characteristic columns: {len(char_cols)}")
print(f"After merging chars + returns: {base_data.shape}")
print(f"Date range: {base_data['date'].min()} to {base_data['date'].max()}")
print(f"Unique stocks: {base_data['permno'].nunique()}")

chars shape: (4117300, 97)
crsp shape: (3352317, 6)
Number of characteristic columns: 94
After merging chars + returns: (3305648, 101)
Date range: 1957-03 to 2021-12
Unique stocks: 25770


In [21]:
# ====================================================
# Merge macro predictors (lagged by 1 month)
# ====================================================

# Lag macro predictors by 1 month (we predict month t returns using month t-1 macro)
macro_predictors_lagged = macro_predictors.copy()
macro_predictors_lagged['date'] = pd.PeriodIndex(macro_predictors_lagged['date'], freq='M') + 1

base_data = base_data.merge(
    macro_predictors_lagged,
    on="date",
    how="left",
    validate="many_to_one"
)

print(f"After merging macro predictors: {base_data.shape}")

# Quick missingness check for macro columns
print("\nMacro missingness (%):")
print((base_data[macro_vars].isnull().mean() * 100).round(4))

After merging macro predictors: (3305648, 109)

Macro missingness (%):
dp_macro      0.0
ep_macro      0.0
bm_macro      0.0
ntis_macro    0.0
tbl_macro     0.0
tms_macro     0.0
dfy_macro     0.0
svar_macro    0.0
dtype: float64


In [22]:
# ====================================================
# Prepare SIC-2 industry code
# ====================================================

base_data["sic2"] = (base_data["siccd"] // 100).astype("Int64")

print("Unique SIC-2 groups in current sample:", base_data["sic2"].nunique())
print("First SIC-2 values:", sorted(base_data["sic2"].dropna().unique())[:15])

Unique SIC-2 groups in current sample: 90
First SIC-2 values: [np.int64(0), np.int64(1), np.int64(2), np.int64(4), np.int64(5), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16)]


In [23]:
# ====================================================
# Cross-sectional rank transformation of raw characteristics 
# NEED TO DOBLE CHECK IF IT IS ALREADY RANKED
# ====================================================


# Ensure numeric type for characteristics   
for c in char_cols:
    base_data[c] = pd.to_numeric(base_data[c], errors="coerce")

print("Applying cross-sectional rank transformation to raw characteristics...")
print("(This may take a while)")

# Cross-sectional ranks per month
ranks = base_data.groupby("date")[char_cols].rank(method="average", na_option="keep")

# Number of valid observations per month and characteristic
n_valid = base_data[char_cols].notna().groupby(base_data["date"]).transform("sum")

# Map ranks into [-1, 1]
ranked_chars = 2 * (ranks - 1).div(n_valid - 1) - 1

# If a month/characteristic has <= 1 valid obs, set to 0
ranked_chars = ranked_chars.mask(n_valid <= 1, 0)

# Fill remaining NaNs with 0
base_data[char_cols] = ranked_chars.fillna(0).astype("float32")

print("Rank transformation complete.")
print("base_data shape:", base_data.shape)

# --- ADD THIS LINE TO FIX THE PARQUET ERROR ---
base_data['date'] = base_data['date'].dt.to_timestamp(how='end')
# ----------------------------------------------

base_data.to_parquet(DATA_DIR / "base_data.parquet", index=False, engine='fastparquet')




Applying cross-sectional rank transformation to raw characteristics...
(This may take a while)
Rank transformation complete.
base_data shape: (3305648, 109)


In [24]:
# ====================================================
# Create industry dummies and interactions from ranked characteristics
# ====================================================

# Industry dummies
industry_dummies = pd.get_dummies(base_data["sic2"], prefix="ind", dtype="int8")

print(f"Number of SIC-2 industry groups: {industry_dummies.shape[1]}")
print(f"Industry dummy columns: {industry_dummies.columns.tolist()[:10]}...")

# Ensure macro vars are numeric and compact
for c in macro_vars:
    base_data[c] = pd.to_numeric(base_data[c], errors="coerce").astype("float32")

# Create interactions using RANKED characteristics
interaction_blocks = []

print("Creating characteristic × macro interactions...")

for macro_var in macro_vars:
    block = base_data[char_cols].mul(base_data[macro_var], axis=0)
    block.columns = [f"{char_col}_x_{macro_var}" for char_col in char_cols]
    block = block.astype("float32")
    interaction_blocks.append(block)
    print(f"Done: {macro_var}")

interactions = pd.concat(interaction_blocks, axis=1)

print(f"Number of firm characteristics: {len(char_cols)}")
print(f"Number of macro variables: {len(macro_vars)}")
print(f"Number of interaction terms: {interactions.shape[1]}")

baseline_feature_count = len(char_cols) + interactions.shape[1] + industry_dummies.shape[1]
print(f"\nBaseline feature count = {len(char_cols)} + {interactions.shape[1]} + {industry_dummies.shape[1]} = {baseline_feature_count}")

# Save components
industry_dummies.to_parquet(DATA_DIR / "industry_dummies.parquet", index=False)
interactions.to_parquet(DATA_DIR / "interactions.parquet", index=False)

print("Saved:", DATA_DIR / "industry_dummies.parquet")
print("Saved:", DATA_DIR / "interactions.parquet")

Number of SIC-2 industry groups: 90
Industry dummy columns: ['ind_0', 'ind_1', 'ind_2', 'ind_4', 'ind_5', 'ind_7', 'ind_8', 'ind_9', 'ind_10', 'ind_11']...
Creating characteristic × macro interactions...
Done: dp_macro
Done: ep_macro
Done: bm_macro
Done: ntis_macro
Done: tbl_macro
Done: tms_macro
Done: dfy_macro
Done: svar_macro
Number of firm characteristics: 94
Number of macro variables: 8
Number of interaction terms: 752

Baseline feature count = 94 + 752 + 90 = 936
Saved: data/industry_dummies.parquet
Saved: data/interactions.parquet


In [25]:
# ====================================================
import gc # Garbage collector

# ====================================================
# Assemble final GKX-style panel
# ====================================================

base_cols = ["permno", "date", "ret_excess", "me", "exchcd"]

# For a GKX-style baseline feature set, do NOT add macro_vars as standalone features.
# Keep: ranked characteristics + characteristic x macro interactions + industry dummies

gkx_panel = pd.concat(
    [
        base_data[base_cols],
        base_data[char_cols],
        interactions,
        industry_dummies,
    ],
    axis=1,
    copy=False,
)

# ---> CRITICAL MEMORY SAVING STEP <---
# Delete the individual dataframes now that they are combined into gkx_panel.
# This instantly frees up about 5-6 GB of RAM before we try to save!
del interactions
del industry_dummies
gc.collect() # Forces Python to instantly empty the memory garbage bin
# -------------------------------------

print(f"GKX panel shape: {gkx_panel.shape}")
print(f"Date range: {gkx_panel['date'].min()} to {gkx_panel['date'].max()}")
print(f"Unique stocks: {gkx_panel['permno'].nunique()}")
print("Missing values (% by column):")
print((gkx_panel.isnull().sum() / len(gkx_panel) * 100).describe())
print(f"Memory usage (GB): {gkx_panel.memory_usage(deep=True).sum() / 1024**3:.2f}")

# ---> WE NO LONGER COPY THE DATAFRAME <---
print("Saving to Parquet (this might take a few minutes)...")

# Save it directly using fastparquet
gkx_panel.to_parquet(DATA_DIR / "gkx2020_panel.parquet", index=False, engine='fastparquet')

print("Saved:", DATA_DIR / "gkx2020_panel.parquet")

GKX panel shape: (3305648, 941)
Date range: 1957-03-31 23:59:59.999999999 to 2021-12-31 23:59:59.999999999
Unique stocks: 25770
Missing values (% by column):
count    9.410000e+02
mean     3.214799e-08
std      9.861627e-07
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      3.025125e-05
dtype: float64
Memory usage (GB): 10.83
Saving to Parquet (this might take a few minutes)...
Saved: data/gkx2020_panel.parquet


---
## 5. Train/Validation/Test Split

Following GKX (2020):
- **Training:** 1957-1974 (18 years)
- **Validation:** 1975-1986 (12 years)
- **Test (OOS):** 1987-2021 (35 years)

The training window expands over time (expanding window scheme).

In [27]:
# ====================================================
# Define sample splits
# ====================================================

# Changed from pd.Period to exact pd.Timestamp dates
train_end = pd.Timestamp("1974-12-31")
valid_end = pd.Timestamp("1986-12-31")
test_end  = pd.Timestamp("2021-12-31")

mask_train = gkx_panel["date"] <= train_end
mask_valid = (gkx_panel["date"] > train_end) & (gkx_panel["date"] <= valid_end)
mask_test  = (gkx_panel["date"] > valid_end) & (gkx_panel["date"] <= test_end)

print(f"Training set:   {mask_train.sum():>10,} obs  (... to {train_end.date()})")
print(f"Validation set: {mask_valid.sum():>10,} obs  (1975 to {valid_end.date()})")
print(f"Test set:       {mask_test.sum():>10,} obs   (1987 to {test_end.date()})")

gkx_panel["split"] = np.select(
    [mask_train, mask_valid, mask_test],
    ["train", "valid", "test"],
    default="outside"
)

print(gkx_panel["split"].value_counts(dropna=False))

Training set:      449,632 obs  (... to 1974-12-31)
Validation set:    713,189 obs  (1975 to 1986-12-31)
Test set:        2,138,514 obs   (1987 to 2021-12-31)
split
test       2138514
valid       713189
train       449632
outside       4313
Name: count, dtype: int64


---
## Summary

| Component | Source | Observations |
|---|---|---|
| 94 firm characteristics | Dacheng Xiu's website (datashare.zip) | Pre-lagged |
| Monthly excess returns | CRSP via WRDS | Adjusted for delistings |
| 8 macro predictors | Welch & Goyal (2008) | Lagged 1 month |
| Industry dummies | SIC-2 codes from CRSP | Cross-sectional |
| Interactions | 94 x 8 = 752 | Char x Macro |

### Next steps for the replication:
1. Train ML models: OLS, Ridge, Lasso, Elastic Net, PCR, PLS, Random Forest, Gradient Boosting, Neural Networks (NN1-NN5)
2. Evaluate using out-of-sample $R^2_{OOS}$
3. Portfolio analysis: decile sorts based on predicted returns
4. Variable importance analysis